In [1]:
%load_ext autoreload
%autoreload 2

In [26]:
import os
from translast.loaders.tokenizer import GenomeIterator, train_sentencepiece, _kmer_split
from translast.config import MODELS_DIR, PROCESSED_DATA_DIR

In [17]:
transcripts = '../data/raw/gencode.v47.transcripts.fa.gz'
raw_transcripts = GenomeIterator(transcripts, 'fasta')

In [27]:
def _kmer_split_bk(k: int, sequence: str):
    return " ".join([sequence[j: j + k] for j in range(len(sequence) - k + 1)])
    
# Byte length to determine the max_sentence_length on sentencepiece
byte_length = len(_kmer_split_bk(17, raw_transcripts[0]).encode('utf-8'))
print(f"kmer profile: \n {_kmer_split_bk(17, raw_transcripts[0])[0:(17+1)*4]}")
print(f"Byte length: {byte_length}")

kmer profile: 
 CGCAGAGACGGGTAGAA GCAGAGACGGGTAGAAC CAGAGACGGGTAGAACC AGAGACGGGTAGAACCT 
Byte length: 24533


In [30]:
# Byte length to determine the max_sentence_length on sentencepiece
byte_length = len(_kmer_split(17, raw_transcripts[0], 'utf-8').encode('utf-8'))
print(f"kmer profile in utf-8 compression: \n {_kmer_split(17, raw_transcripts[0], 'utf-8')[0:(17+1)*4]}")
print(f"Byte length: {byte_length}")

kmer profile in utf-8 compression: 
`  ],5    X  Kj@ ! ]  t  jK   ],  t @ K
Byte length: 5169


In [ ]:
seq_tokenizer = train_sentencepiece(raw_transcripts,
                                    google = True,
                                    out = 'transcripts_sentencepiece',
                                    name = 'k17_unigram',
                                    vocab_size = 138648, k=17, fast = True)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input_format: 
  model_prefix: 
  model_type: UNIGRAM
  vocab_size: 138648
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 1000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 391697
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 17
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: [CLS]
  user_defined_symbols: [SEP]
  user_defined_symbols: [MASK]
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 1
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: 3
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_pi

In [6]:
# Open a file in write mode
with open('gencode.v47.transcripts.k17.txt', 'w') as file:
    # Write each string from the iterator to the file, line by line
    for line in map(lambda seq: _kmer_split(17, seq), raw_transcripts):
        file.write(line + '\n')

## Convert SPM tokenizer to PreTrainedTokenizerFast

In [9]:
from tokenizers import BertWordPieceTokenizer, SentencePieceUnigramTokenizer
from transformers import PreTrainedTokenizerFast, PreTrainedTokenizer, convert_slow_tokenizer

import sentencepiece as spm

from typing import Union, List

def convert_tokens_to_ids(spm_tokenizer, tokens: Union[str, List[str]]) -> Union[int, List[int]]:
    if tokens is None:
        return None
    if isinstance(tokens, str):
        return spm_tokenizer.piece_to_id(tokens)
    ids = []
    for token in tokens:
        ids.append(spm_tokenizer.piece_to_id(token))
    return ids

def load_tokenizer(tokenizer_path):
    tokenizer = AlbertTokenizer.from_pretrained(tokenizer_path)
    return tokenizer

# Define special tokens
special_tokens = {
    'unk_token': '<unk>',
    'sep_token': '[SEP]',
    'cls_token': '[CLS]',
    'pad_token': '<pad>',
    'mask_token': '[MASK]'
}

model_name = 'gencode.v47.transcripts.k17'
model_dir = 'transcripts_sentencepiece/unigram'
model_max_length=1024

spm_tokenizer = spm.SentencePieceProcessor(model_file=os.path.join(model_dir, 'google', f'{model_name}.model'))
spm_tokenizer.vocab_file = os.path.join(model_dir, 'google', f'{model_name}.model')
spm_tokenizer.keep_accents = True
spm_tokenizer.do_lower_case = True
spm_tokenizer.convert_tokens_to_ids = lambda t: convert_tokens_to_ids(spm_tokenizer, t)

# Create a custom tokenizer using the SentencePiece model
arbert_tokenizer = convert_slow_tokenizer.AlbertConverter(spm_tokenizer)
arbert_tokenizer = arbert_tokenizer.converted()

# Save the tokenizer
arbert_tokenizer.save(os.path.join(model_dir, 'huggingface', 'slow', f'{model_name}.json'))

In [10]:
fast_tokenizer = PreTrainedTokenizerFast(tokenizer_file= os.path.join(model_dir, 'huggingface', 'slow', f'{model_name}.json'),
                                   model_max_length=model_max_length, padding_side='right', truncation_side='right',
                                   local_files_only=True, **special_tokens)
fast_tokenizer.save_pretrained(os.path.join(model_dir, 'huggingface', model_name))

('transcripts_sentencepiece/unigram/huggingface/gencode.v47.transcripts.k17/tokenizer_config.json',
 'transcripts_sentencepiece/unigram/huggingface/gencode.v47.transcripts.k17/special_tokens_map.json',
 'transcripts_sentencepiece/unigram/huggingface/gencode.v47.transcripts.k17/tokenizer.json')

In [31]:
fast_tokenizer.is_fast

True

In [15]:
fast_tokenizer.get_special_tokens_mask

<bound method PreTrainedTokenizerBase.get_special_tokens_mask of PreTrainedTokenizerFast(name_or_path='', vocab_size=138648, model_max_length=1024, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '<unk>', 'sep_token': '[SEP]', 'pad_token': '<pad>', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=False),
	5: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=Fal